# Mesa Data Preparation
So, I've finally decided that there's no way to circumvent the need to use MESA (Modules for Experimental Stellar Astrophysics) to do this research.

This means we will have to prepare the initial simulation data for MESA. We retrieved the stars from the Gaia DR3 dataset using the following ADQL query:

```sql
SELECT TOP 1000000
	gaia.source_id, 
	gaia.ra as ra,
	gaia.dec as dec,
	parameters.mass_flame AS mass,
	parameters.radius_gspphot AS radius,
	parameters.age_flame as age
FROM gaiadr3.astrophysical_parameters AS parameters
INNER JOIN gaiadr3.gaia_source as gaia
ON gaia.source_id = parameters.source_id
WHERE 
	parameters.mass_flame IS NOT NULL
	AND parameters.fem_gspspec IS NOT NULL
	AND parameters.age_flame IS NOT NULL
	AND parameters.evolstage_flame > 100 -- 100 = Zero age main sequence star
	AND parameters.evolstage_flame < 360 -- 360 = main sequence turn off
```

## The Simulations
We will be running an evolutionary simulation for each star up until its current age (estimated by Gaia).
We will provide metallicity, initial mass, age, and initial helium (calculated from the metallicity) as inputs to the simulation.

We will export the simulation parameters to a csv `data.csv` file and use that as input for parallel execution using GNU Parallel.
The data will be exported to the `parallel` directory which will then be used to run the simulations.

In [3]:
# Import libraries
import pandas as pd
import numpy as np
from astropy.table import Table
from astropy.io import fits
import os
import requests
import gzip
from collections import namedtuple
from typing import Union
from pathlib import Path

In [54]:
masses = np.arange(0.8, 1.1, 0.05) # 7 entries
helium_fracs = np.arange(0.22, 0.36, 0.005) # 28 Entries

headers = ['mass', 'y']
rows = []

for mass in masses:
    for y in helium_fracs:
        rows.append([mass, y])

dataset = pd.DataFrame(rows, columns=headers)
dataset

,mass,y
0,0.8,0.220
1,0.8,0.225
2,0.8,0.230
3,0.8,0.235
4,0.8,0.240
...,...,...
191,1.1,0.335
192,1.1,0.340
193,1.1,0.345
194,1.1,0.350


In [55]:
# Export the data to a csv file
dataset.to_csv('simulate/data.csv')

### Setting up for simulations
Please be aware that you will need to download and install both GNU Parallel and MESA.

You can install GNU Parallel using `sudo dnf install parallel` or `sudo apt install parallel`, depending on your distribution.

Please follow the instructions on [MESA's website](https://docs.mesastar.org/en/latest/installation.html).

I used SDK Version **23.7.3** and MESA Version **24.08.1**.

I would highly recommend downloading the source directly from the [MESA GitHub](https://github.com/MESAHub/mesa/releases/tag/r24.03.1) 
rather than downloading it from Zenodo since it can be **slow** to download from Zenodo.

I used a **Fedora 40** distribution tailored for astronomy. I downloaded the ISO image from [here](https://fedoraproject.org/labs/astronomy/).